In [ ]:
import numpy as np  
from amuse.lab import *
from matplotlib import pyplot as plt


In [ ]:
def init_cluster(N, R):
    masses = new_salpeter_mass_distribution(N, 0.1 | units.MSun, 100 | units.MSun)
    converter = nbody_system.nbody_to_si(masses.sum(), R)
    bodies = new_plummer_model(N, convert_nbody=converter)
    bodies.mass = masses
    bodies.scale_to_standard(converter)
    return bodies, converter

def plot_cluster(bodies):
    # plt.figure(figsize=(5,5))
    plt.scatter(bodies.x.value_in(units.m), bodies.y.value_in(units.m))
    # plt.xlabel("x [m]")
    # plt.ylabel("y [m]")
    # plt.title("Star Cluster")
    # plt.axis('equal')
    # plt.grid()
    # plt.show()


In [ ]:
np.random.seed(1234)
bodies, converter = init_cluster(10, 0.1| units.pc)
bodies_init = bodies.copy()
plot_cluster(bodies_init)


In [ ]:
converter = nbody_system.nbody_to_si(bodies.mass.sum(), np.mean(bodies.position.lengths()))

plt.figure(figsize=(5,5))
plt.scatter(bodies.x.value_in(units.m), bodies.y.value_in(units.m), label="t=0")


gravity = ph4(convert_nbody=converter)
gravity.particles.add_particles(bodies)

channels = {}
channels['to_gravity'] = bodies.new_channel_to(gravity.particles)
channels['from_gravity'] = gravity.particles.new_channel_to(bodies)

time =0 | units.yr
dt = 0.1 | units.Myr
time_end = 1 | units.Myr

while time < time_end:
    time += dt
    gravity.evolve_model(time)
    channels['from_gravity'].copy()
    
plt.scatter(bodies.x.value_in(units.m), bodies.y.value_in(units.m), label=f"t={time.value_in(units.Myr):.1f} Myr")
plt.xlabel("x [m]")
plt.ylabel("y [m]")
plt.title("Star Cluster")
plt.axis('equal')
plt.legend()
plt.grid()
plt.show()
